# Macro-Financial Risk Forecasting with ARIMAX-GARCH models


#### Modeling volatility regimes and macroeconomic impacts on financial risk factors using Python. 

|Macro Variables | Sources |
|---|---|
| USD/MXN | Yahoo Finance |
| Inflation | FRED |
| Banxico Rate | Banxico API |
| Oil Prices | FRED |
| VIX | Yahoo Finance |
| Fed Funds Rate | FRED |
| Industrial Activity | FRED (MEXPROINDMISMEI) |

# Chapter 1 - Acquiring Financial Data

### 1.1 Install the libraries:

In [6]:
!pip install yfinance fredapi pandas requests

In [34]:
!pip install pandas_datareader

### 1.2. Import the libraries

In [8]:
import pandas as pd
import yfinance as yf
import requests
from fredapi import Fred

### 1.3. Download MXN/USD variable. 

In [98]:
mxn_usd = yf.download(
    "mxn=X",
    start="2008-01-01",
    end="2025-01-01",
    progress=False,
    auto_adjust=False
)

# Keep only closing prices
mxn_usd = mxn_usd[["Close"]]

# Rename column
mxn_usd.columns = ["mxn_usd"]

# Verify time range
print(mxn_usd.index.min())
print(mxn_usd.index.max())

mxn_usd.head()

2008-01-01 00:00:00
2024-12-31 00:00:00


,mxn_usd
2008-01-01,10.8770
2008-01-02,10.8995
2008-01-03,10.8701
2008-01-04,10.9240
2008-01-07,10.8916


### 1.4  Download VIX 

In [43]:
vix = yf.download(
    "^VIX",
    start="2008-01-01",
    end="2025-01-01",
    progress=False,
    auto_adjust=False
)

# Keep only closing prices
vix = vix[["Close"]]

# Rename column
vix.columns = ["vix"]

# Verify time range
print(vix.index.min())
print(vix.index.max())

vix.head()

2008-01-02 00:00:00
2024-12-31 00:00:00


,vix
2008-01-02,23.170000
2008-01-03,22.490000
2008-01-04,23.940001
2008-01-07,23.790001
2008-01-08,25.430000


### 1.5  FRED API KEY 

In [22]:
FRED_API_KEY = "0a98ea864b42d1d27ad7daa75c85005a"

fred = Fred(api_key=FRED_API_KEY)

### 1.6  Download Oil Prices (WTI).

In [90]:
oil = fred.get_series("DCOILWTICO")

# Restrict time range
oil = oil.loc["2008-01-01":"2025-01-01"]

# Convert to DataFrame (to get a tabular structure)
oil = oil.to_frame(name="oil_prices")

# Verify time range
print(oil.index.min())
print(oil.index.max())

oil.head()

2008-01-01 00:00:00
2025-01-01 00:00:00


,oil_prices
2008-01-01,NaN
2008-01-02,99.64
2008-01-03,99.17
2008-01-04,97.90
2008-01-07,95.08


### 1.7 Download Fed Funds Rate. 

In [45]:
fed_rate = fred.get_series("FEDFUNDS")

# Restrict time range
fed_rate = fed_rate.loc["2008-01-01":"2025-01-01"]

# Convert to DataFrame
fed_rate = fed_rate.to_frame(name="fed_funds_rate")

# Verify time range
print(fed_rate.index.min())
print(fed_rate.index.max())

fed_rate.head()

2008-01-01 00:00:00
2025-01-01 00:00:00


,fed_funds_rate
2008-01-01,3.94
2008-02-01,2.98
2008-03-01,2.61
2008-04-01,2.28
2008-05-01,1.98


### 1.8 Download Industrial Activity( México). 

In [46]:
import pandas_datareader.data as web

industrial_activity = web.DataReader(
    "IPB50001N",
    "fred",
    start="2008-01-01",
    end="2025-01-01"
)

# Rename column
industrial_activity.columns = ["industrial_activity"]

# Verify time range
print(industrial_activity.index.min())
print(industrial_activity.index.max())

industrial_activity.head()

2008-01-01 00:00:00
2025-01-01 00:00:00


,industrial_activity
DATE,
2008-01-01,100.9807
2008-02-01,100.7289
2008-03-01,101.0713
2008-04-01,99.7489
2008-05-01,99.4369


### 1.9 Download México CPI 

In [83]:
cpi= fred.get_series("MEXCPIALLMINMEI")

# Restrict time range( we take it from 2007 - 2025)
cpi = cpi.loc["2007-01-01":"2025-01-01"]    


# Convert to DataFrame
cpi = cpi.to_frame(name="CPI")

# Verify time range
print(cpi.index.min())
print(cpi.index.max())

cpi.head()

2007-01-01 00:00:00
2024-07-01 00:00:00


,CPI
2007-01-01,71.89153
2007-02-01,72.09248
2007-03-01,72.24850
2007-04-01,72.20536
2007-05-01,71.85311


### 1.9 Inflation rate derivated from CPI 

$$
\pi_t =
\left(
\frac{CPI_t}{CPI_{t-12}} - 1
\right)\times100
$$

In [84]:
# Inflation Year-over-Year (%)
cpi["inflation_rate"] = (
    cpi["CPI"].pct_change(12)
) * 100

# Keep only inflation rate and analysis period
inflation_rate = cpi[["inflation_rate"]].loc["2008-01-01":"2025-01-01"]

inflation_rate.head()

,inflation_rate
2008-01-01,3.704372
2008-02-01,3.722732
2008-03-01,4.248877
2008-04-01,4.548540
2008-05-01,4.947566


### 1.10 Download Banxico Interest Rate. 

In [68]:
BANXICO_TOKEN = "c1b8142a5f13f80cb16992aa2972db3352ebc72aaec7e554ce33b35909e4cae3"

url = "https://www.banxico.org.mx/SieAPIRest/service/v1/series/SF61745/datos"

headers = {
    "Bmx-Token": BANXICO_TOKEN
}

response = requests.get(url, headers=headers)

data = response.json()

banxico_rate = pd.DataFrame(
    data['bmx']['series'][0]['datos']
)

# Rename columns
banxico_rate.columns = ["date", "banxico_rate"]

# Convert date column
banxico_rate["date"] = pd.to_datetime(
    banxico_rate["date"],
    format="%d/%m/%Y"
)

# Set index
banxico_rate.set_index("date", inplace=True)

# Convert values to numeric
banxico_rate["banxico_rate"] = pd.to_numeric(
    banxico_rate["banxico_rate"],
    errors="coerce"
)

# Restrict time range
banxico_rate = banxico_rate.loc["2008-01-01":"2025-01-01"]

# Verify time range
print(banxico_rate.index.min())
print(banxico_rate.index.max())

banxico_rate.head()

2008-01-21 00:00:00
2025-01-01 00:00:00


,banxico_rate
date,
2008-01-21,7.5
2008-01-22,7.5
2008-01-23,7.5
2008-01-24,7.5
2008-01-25,7.5


### 1.11. Merge all dataset. 

In [99]:
df = pd.concat([
    mxn_usd,
    vix,
    oil,
    fed_rate,
    industrial_activity,
    inflation_rate,
    banxico_rate
], axis=1)

df.head()

,mxn_usd,vix,oil_prices,fed_funds_rate,industrial_activity,inflation_rate,banxico_rate
2008-01-01,10.8770,NaN,NaN,3.94,100.9807,3.704372,NaN
2008-01-02,10.8995,23.170000,99.64,NaN,NaN,NaN,NaN
2008-01-03,10.8701,22.490000,99.17,NaN,NaN,NaN,NaN
2008-01-04,10.9240,23.940001,97.90,NaN,NaN,NaN,NaN
2008-01-07,10.8916,23.790001,95.08,NaN,NaN,NaN,NaN


### 1.12  Create folders:

In [91]:
import os

# Create folders if they do not exist
os.makedirs("../data/raw", exist_ok=True)

1.13. Save CSV, data raw:

In [92]:
df.to_csv("../data/raw/macro_financial_data.csv")

Note: The inflation series was obtained from FRED at a monthly frequency. When merged with daily financial variables such as MXN/USD, VIX, and oil prices, missing values naturally emerged because inflation data are only released once per month. These missing observations do not represent data quality issues but rather differences in reporting frequency across variables.